# Imports 

In [ ]:
## Essential Imports: 
import os
import numpy as np
import qp
import tables_io
from pathlib import Path 
from pzflow.examples import get_galaxy_data
import ceci

## RAIL-Specific Imports: 
import rail


import rail.creation 
import rail.creation.engines
from rail.creation.engines.flowEngine import FlowModeler, FlowCreator, FlowPosterior
from rail.core.data import TableHandle
from rail.core.stage import RailStage

from rail.estimation.algos.train_z import TrainZEstimator, TrainZInformer
from rail.estimation.algos.pzflow_nf import PZFlowInformer, PZFlowEstimator   


from rail.evaluation.evaluator import Evaluator


## Data Storage: 
DS = RailStage.data_store
DS.__class__.allow_overwrite = True


### CMNN, PZFlow, FlexZBoost, GPZ, trainz for control

In [ ]:
## these don't work in rail_6-4-24, but they do in rail---new


from rail.creation.degradation.lsst_error_model import LSSTErrorModel
from rail.creation.degradation.spectroscopic_degraders import InvRedshiftIncompleteness
from rail.creation.degradation.quantityCut import QuantityCut 
from rail.core.utilStages import ColumnMapper, TableConverter
from rail.estimation.algos.cmnn import Inform_CMNNPDF, CMNNPDF
from rail.estimation.algos.gpz import GPzInformer, GPzEstimator 
from rail.estimation.algos.flexzboost import FlexZBoostInformer, FlexZBoostEstimator  

In [ ]:
#from rail.stages import *
#rail.stages.import_and_attach_all()
#for val in RailStage.pipeline_stages.values():
#    print(val[0])

In [ ]:
import numpy as np

def removeNans(df): 
    lsu = df.index[np.isnan(df['mag_u_lsst']) ==True].tolist() 
    lsg = df.index[np.isnan(df['mag_g_lsst']) ==True].tolist() 
    lsr = df.index[np.isnan(df['mag_r_lsst']) ==True].tolist() 
    lsi = df.index[np.isnan(df['mag_i_lsst']) ==True].tolist() 
    lsz = df.index[np.isnan(df['mag_z_lsst']) ==True].tolist() 
    lsy = df.index[np.isnan(df['mag_y_lsst']) ==True].tolist() 

    nans = list(set(lsu+lsg+lsr+lsi+lsz+lsy))
    new_df = df.drop(nans)

    return new_df 

In [ ]:
def magCut(df, mag):
    lsu = df.index[df['mag_u_lsst'] > mag].tolist() 
    lsg = df.index[df['mag_g_lsst'] > mag].tolist() 
    lsr = df.index[df['mag_r_lsst'] > mag].tolist() 
    lsi = df.index[df['mag_i_lsst'] > mag].tolist() 
    lsz = df.index[df['mag_z_lsst'] > mag].tolist() 
    lsy = df.index[df['mag_y_lsst'] > mag].tolist() 

    cuts = list(set(lsu+lsg+lsr+lsi+lsz+lsy))
    new_df = df.drop(cuts)

    return new_df 

In [ ]:
def nansList(df): 
    lsu = df.index[np.isnan(df['mag_u_lsst']) ==True].tolist() 
    lsg = df.index[np.isnan(df['mag_g_lsst']) ==True].tolist() 
    lsr = df.index[np.isnan(df['mag_r_lsst']) ==True].tolist() 
    lsi = df.index[np.isnan(df['mag_i_lsst']) ==True].tolist() 
    lsz = df.index[np.isnan(df['mag_z_lsst']) ==True].tolist() 
    lsy = df.index[np.isnan(df['mag_y_lsst']) ==True].tolist() 

    nans = list(set(lsu+lsg+lsr+lsi+lsz+lsy))

    return nans

# Model

In [ ]:
# import pandas as pd 
# simpath = "../paper_data/data/rubin_roman_cat_galaxy_10307.parquet"
# simdf = pd.read_parquet(simpath)
# simdf


In [ ]:
# zero_point_dict ={"u":12.627880975839947,"g":14.486360566315488,
# "r":14.324886825965237,"i":13.998598224053055,
# "z":13.612736383992512,"y":12.785119907991263}

# def flux_to_mag(flux, band):
#     zero = zero_point_dict[band]
#     return zero - 2.5*np.log10(flux)

In [ ]:
# u_band_flux = flux_to_mag(simdf['lsst_flux_u'], 'u')
# g_band_flux = flux_to_mag(simdf['lsst_flux_g'], 'g')
# r_band_flux = flux_to_mag(simdf['lsst_flux_r'], 'r')
# i_band_flux = flux_to_mag(simdf['lsst_flux_i'], 'i')
# z_band_flux = flux_to_mag(simdf['lsst_flux_z'], 'z')
# y_band_flux = flux_to_mag(simdf['lsst_flux_y'], 'y')

In [ ]:
# simdf.insert(10, "mag_u_lsst", u_band_flux, True)
# simdf.insert(11, "mag_g_lsst", g_band_flux, True)
# simdf.insert(12, "mag_r_lsst", r_band_flux, True)
# simdf.insert(13, "mag_i_lsst", i_band_flux, True)
# simdf.insert(14, "mag_z_lsst", z_band_flux, True)
# simdf.insert(15, "mag_y_lsst", y_band_flux, True)
# simdf

In [ ]:
# sim_outpath = sim_outpath = "../paper_data/data/rubin_roman_catalog.pq"
# simdf.to_parquet(sim_outpath)

In [ ]:
# import pandas as pd
# simdf = pd.read_parquet("../paper_data/data/rubin_roman_catalog.pq")

In [ ]:
# ibanddf = magCut(simdf, 25.3)
# ibandinds = ibanddf.index.tolist()

# # importing random module
# import random
# # getting numbers from 0 to 100
# inputNumbers = ibandinds
# # printing 4 random numbers from the given range of numbers which are
# # non-repeating using sample() function
# print("non-repeating random numbers are:")
# rands = list(set(random.sample(inputNumbers, 169126)))
# iband_cut_df = ibanddf.drop(rands)

In [ ]:
# import numpy as np

# # importing random module
# import random
# # getting numbers from 0 to 100
# inputNumbers = indls
# # printing 4 random numbers from the given range of numbers which are
# # non-repeating using sample() function
# print("non-repeating random numbers are:")
# rands = list(set(random.sample(inputNumbers, 399349)))
# cut_df = trimdf.drop(rands)

# import matplotlib.pyplot as plt

# plt.scatter(simdf['redshift'], simdf['mag_g_lsst'], s=0.5)
# plt.scatter(cut_df['redshift'], cut_df['mag_g_lsst'], s=0.5)


In [ ]:
os.chdir('../plots/Photo-z-Stress-Test')

def makeModel():
    #path to access the data 
    DATA_DIR =  Path().resolve() / 'data' #"../paper_data/data" #"Path().resolve() / "data"
    #DATA_DIR.mkdir(exist_ok=True)

    print(type(DATA_DIR))

    catalog_file = DATA_DIR / "1e6_trimmed_rubin_roman_catalog.pq" #"base_catalog.pq"

    print(type(catalog_file))
    print(catalog_file)

    bands = ['u','g','r','i','z','y']
    band_dict = {band:f'mag_{band}_lsst' for band in bands}
    band_dict_err = {f'mag_{band}_lsst_err':f'mag_err_{band}_lsst' for band in bands}
    band_dict_err = {f'mag_err_{band}_lsst' for band in bands}
    
    #array of galaxies w/ 7 attributes for each: redshift & ugrizy
    # catalog = get_galaxy_data().rename(band_dict, axis=1) 

    #turns array into a table 
    # tables_io.write(catalog, str(catalog_file.with_suffix("")), catalog_file.suffix[1:])

    catalog_file = str(catalog_file)
    flow_file = str(DATA_DIR / "1e6_trained_flow.pkl")

    print(flow_file)

    #we set up the stage 
    flow_modeler_params = {
        "name": "flow_modeler",
        "input": catalog_file,
        "model": flow_file,
        "seed": 0,
        "phys_cols": {"redshift": [0, 3]},
        "phot_cols": {
            "mag_u_lsst": [17, 28],#35],
            "mag_g_lsst": [16, 28],#32],
            "mag_r_lsst": [15, 28], #30],
            "mag_i_lsst": [15, 28], # 30],
            "mag_z_lsst": [14, 28], # 29],
            "mag_y_lsst": [14, 28],
        },
        "calc_colors": {"ref_column_name": "mag_i_lsst"},
    }
    flow_modeler = FlowModeler.make_stage(**flow_modeler_params)
    return flow_modeler, flow_file ##.get_handle("model")

In [ ]:
# !open ~/miniforge3/envs/rail_6-4-24/lib/python3.10/site-packages/rail/creation/engines/

In [ ]:
# !open ~/miniforge3/envs/rail_6-4-24/lib/python3.10/site-packages/rail/estimation/

In [ ]:
## try to figure out where the estimation directory for this env is from the line above... [...]/rail/estimation/ [stuff] ? 

In [ ]:
modelData, flow_file = makeModel() 

In [ ]:
## trimmed initial dataset at 28th magnitude, filtered to i<25.3 after degradation

In [ ]:
# modelData.fit_model()

# Make Training Set and Test Set 

In [ ]:
def spec_trainSet(ntrain, seed):
    data = FlowCreator.make_stage(
            name = 'spec_train_set',
            model = flow_file,
            n_samples = ntrain,
            seed = seed 
    )
    return data 

def invz_trainSet(ntrain, seed):
    data = FlowCreator.make_stage(
            name = 'invz_train_set',
            model = flow_file,
            n_samples = ntrain,
            seed = seed 
    )
    return data 


spec_train_data = spec_trainSet(1e6, 17) 
invz_train_data = invz_trainSet(1e5, 17)

spec_train_data_2 = invz_trainSet(1e6, 7)

In [ ]:
def testSet(ntest, seed):
    data = FlowCreator.make_stage(
            name = 'test_set',
            model = flow_file,
            n_samples = ntest,
            seed = seed 
    )
    return data #.sample(ntest, seed)

def control_testSet(ntest, seed):
    data = FlowCreator.make_stage(
            name = 'control_test_set',
            model = flow_file,
            n_samples = ntest,
            seed = seed 
    )
    return data #.sample(ntest, seed)

test_data = testSet(1e5, 39)
control_train_data = control_testSet(1e5, 39)

# Degraders

## Inverse Redshift Incompleteness

In [ ]:
def invRedshift(pivot = 1.0):
    assert type(pivot) == float 
    degr = InvRedshiftIncompleteness.make_stage(
        name = 'inv_redshift',
        pivot_redshift = pivot
    )
    return degr 


## LSST Error 

In [ ]:
bands = ['u','g','r','i','z','y']
band_dict = {band:f'mag_{band}_lsst' for band in bands}

def lsstError(dict, seed): 
    deg = LSSTErrorModel.make_stage(
        name='lsst_error',
        renameDict= dict, 
        ndFlag=np.nan,
        seed=seed,
    )
    return deg 

def lsstError_0(dict, seed): 
    deg = LSSTErrorModel.make_stage(
        name='lsst_error_0',
        renameDict= dict, 
        ndFlag=np.nan,
        seed=seed
    )
    return deg 

def lsstError_null(dict, seed): 
    deg = LSSTErrorModel.make_stage(
        name = 'lsst_error_null', 
        renameDict= dict, 
        ndFlag=np.nan,
        seed=seed,
        nYrObs = 100000000,
        tvis = 100000000,
        sigmaSys= 0.000
    ) 
    return deg

# def lsstError_control(dict, seed): 
#     deg = LSSTErrorModel.make_stage(
#         name='lsst_error_control',
#         renameDict= dict, 
#         ndFlag=np.nan,
#         seed=seed,
#     )
#     return deg 

# Quantity Cuts

In [ ]:
# write a dictionary with the different bands and magnitudes you want",


def quantCuts_1(cuts_dict):
    quantity_cut = QuantityCut.make_stage(
        name='quantity_cut_1',    
        cuts=cuts_dict,
    )
    return quantity_cut

def quantCuts_2(cuts_dict):
    quantity_cut = QuantityCut.make_stage(
        name='quantity_cut_2',    
        cuts=cuts_dict,
    )
    return quantity_cut

qcuts_dict = {#'mag_u_lsst': [...],
            #'mag_g_lsst': [...], 
            #'mag_r_lsst': [...], 
            'mag_i_lsst': 25.3,
            #'mag_z_lsst': [...], 
            #'mag_y_lsst': [...], 
            'redshift': [0.0, 3.0]
            }

## Survey-Based Degraders

In [ ]:
from rail.creation.degradation.spectroscopic_selections import *

def specSelectBOSS(ntrain):
    degr = SpecSelection_BOSS.make_stage(
        name = 'specselection_boss',
        N_tot = ntrain
    )
    return degr 

def specSelectDEEP2(ntrain):
    degr = SpecSelection_DEEP2.make_stage(
        name = 'specselection_deep2',
        N_tot = ntrain
    )
    return degr 

def specSelectGAMA(ntrain):
    degr = SpecSelection_GAMA.make_stage(
        name = 'specselection_gama',
        N_tot = ntrain
    )
    return degr 

def specSelectHSC(ntrain):
    degr = SpecSelection_HSC.make_stage(
        name = 'specselection_HSC',
        N_tot = ntrain
    )
    return degr 

def specSelectVVDSf02(ntrain):
    degr = SpecSelection_VVDSf02.make_stage(
        name = 'specselection_VVDSf02',
        N_tot = ntrain
    )
    return degr 

def specSelectzCOSMOS(ntrain):
    degr = SpecSelection_zCOSMOS.make_stage(
        name = 'specselection_zCOSMOS',
        N_tot = ntrain
    )
    return degr 

In [ ]:
# spec_dict = {'BOSS': specSelectBOSS, 
#              'DEEP2': specSelectDEEP2, 
#              'GAMA': specSelectGAMA,
#              'HSC': specSelectHSC, 
#              'VVDSf02': specSelectVVDSf02, 
#              'zCOSMOS': specSelectzCOSMOS } 

# Column Remappers

In [ ]:
bands = ['u','g','r','i','z','y']
band_dict_err = {f'mag_{band}_lsst_err':f'mag_err_{band}_lsst' for band in bands}

def colRemapper_1(dict):
    col_remap = ColumnMapper.make_stage(
    name='col_remapper_1', 
    columns=dict,
    )
    return col_remap

def colRemapper_2(dict):
    col_remap = ColumnMapper.make_stage(
    name='col_remapper_2', 
    columns=dict,
    )
    return col_remap

def tableConverter():
    table_conv = TableConverter.make_stage(
    name='table_conv', 
    output_format='numpyDict',
    )
    return table_conv

# Inform & Estimate 

In [ ]:
def informTrainZ(deg):
    inf = TrainZInformer.make_stage(
    name = 'inform_TrainZ',
    model = deg+'_trainz.pkl',
    hdf5_groupname=""
    )
    return inf

def estimateTrainZ(deg):
    est = TrainZEstimator.make_stage(
    name = 'estimate_TrainZ',
    model = deg+'_trainz.pkl', 
    hdf5_groupname=""
    )
    return est

In [ ]:
def informCMNN(deg):
    inf = Inform_CMNNPDF.make_stage(
    name = 'inform_CMNN',
    model = deg+'_cmnn.pkl',
    hdf5_groupname=""
    )
    return inf

def estimateCMNN(deg):
    est = CMNNPDF.make_stage(
    name = 'estimate_CMNN',
    model = deg+'_cmnn.pkl', 
    hdf5_groupname=""
    )
    return est

In [ ]:
def informGPz(deg):
    inf = GPzInformer.make_stage(
    name = 'inform_GPz',
    model = deg+'_gpz.pkl',
    gpz_method = 'GL',
    hdf5_groupname=""
    )
    return inf

def estimateGPz(deg):
    est = GPzEstimator.make_stage(
    name = 'estimate_GPz',
    model = deg+'_gpz.pkl', 
    gpz_method = 'GL',
    hdf5_groupname=""
    )
    return est

In [ ]:
def informFZBoost(deg):
    info = FlexZBoostInformer.make_stage(
    name ='inform_FZBoost', 
    model = deg+'_fzboost.pkl', 
    hdf5_groupname='',
    )
    return info

def estimateFZBoost(deg):
    est = FlexZBoostEstimator.make_stage(
    name='estimate_FZBoost', 
    nondetect_val=np.nan,
    model= deg+'_fzboost.pkl', 
    hdf5_groupname='',
    # aliases=dict(input='test_data', output='fzboost_estim'),
    nzbins = 100 
    )
    return est 

## PZFlow

In [ ]:
# import os
# import matplotlib.pyplot as plt
# import numpy as np
# %matplotlib inline 
# import pandas as pd
# from astropy.table import QTable, Table, Column
# from collections import OrderedDict

# import rail
# import qp
# from rail.core.data import TableHandle
# from rail.core.stage import RailStage

# DS = RailStage.data_store
# DS.__class__.allow_overwrite = True

In [ ]:
# dir = "../plots/src"
# os.chdir(dir)

# from rail.estimation.algos.pzflow_nf_default import PZFlowInformer, PZFlowEstimator

# pzflow_dict = dict(hdf5_groupname='', output_mode = 'not_default', )


In [ ]:
# def informPZFlow():
#     inf = PZFlowInformer.make_stage(
#     name = 'inform_PZFlow', 
#     model = 'pzflow.pkl', 
#     num_training_epochs = 200,
#     **pzflow_dict
#     )
#     inf._aliases = dict()
#     return inf 

# def estimatePZFlow():
#     est = PZFlowEstimator.make_stage(
#     name='estimate_pzflow',
#     model='pzflow.pkl',
#     **pzflow_dict, 
#     chunk_size = 20000
#     )
#     est._aliases = dict()
#     return est 

In [ ]:
# os.chdir("../paper_data/specSelection_lsstErr_PZFlow/outputs/BOSS/")
# trainFile = DS.read_file("training_data", TableHandle, "./output_lsst_error_0.pq")

# informPZFlow().inform(trainFile)

In [ ]:
# os.chdir("../paper_data/invz_lsstErr_PZFlow/outputs/1.0/")
# testFile = DS.read_file('test_data', TableHandle, "./output_lsst_error.pq")
# estimatePZFlow().estimate(testFile)

# Posteriors 

In [ ]:
# os.chdir("../paper_data/")
# extra_gama = specSelectGAMA(1e6)
# extra_gama.connect_input(spec_train_data)

# gama_2 = extra_gama(spec_train_data.sample(1e6, 12), seed=12)

# extra_lsstErr = lsstError(band_dict, 7)
# lsst_gama_2 = extra_lsstErr(gama_2)

# extra_qCut = quantCuts_1(qcuts_dict)
# gama_qcut_2 = extra_qCut(lsst_gama_2)

In [ ]:
## make test data 
os.chdir("../paper_data/")

lsstErr = lsstError(band_dict, 39)
lsstErr.connect_input(test_data)

qCut = quantCuts_1(qcuts_dict)

lsstErr_test_data = lsstErr(test_data.sample(1e5, 39), seed=39)
cut_data = qCut(lsstErr_test_data)


In [ ]:
import pandas as pd

uncut = pd.read_parquet("../paper_data/output_lsst_error.pq")
cut = pd.read_parquet("../paper_data/output_quantity_cut_1.pq")

In [ ]:
uncut

In [ ]:
cut

In [ ]:
# import tables_io
# import qp
# import numpy as np
# import h5py
# import os
# import matplotlib.pyplot as plt
# import pandas as pd

# os.chdir("../plots/Photo-z-Stress-Test")

# lsstErr_testSet_posts = FlowPosterior.make_stage(name='lsstErr_test_data_posts', 
#                                              column='redshift',
#                                              grid = np.linspace(0, 3.0, 101),
#                                              model=flow_file,
#                                              data = "../paper_data/output_quantity_cut_1.pq"
# )

# # lsstErr_test_pdfs = lsstErr_testSet_posts.get_posterior(lsstErr_deg_data, column='redshift')

# # post_file = .read("../paper_data/output_lsst_error (for posts).pq")

# posts_file = DS.read_file('name', TableHandle, path="../paper_data/output_quantity_cut_1.pq")

# lsstErr_test_pdfs = lsstErr_testSet_posts.get_posterior(posts_file, column = 'redshift')

In [ ]:
# import tables_io
# import qp
# import numpy as np
# import h5py
# import os
# import matplotlib.pyplot as plt
# import pandas as pd

# os.chdir("../paper_data/PZFlow/control")

# testSet_posts = FlowPosterior.make_stage(name='PZFlow_control_data_posts', 
#                                              column='redshift',
#                                              grid = np.linspace(0, 3.0, 101),
#                                              model=flow_file,
#                                              data = "../paper_data/PZFlow/control/control_no_nans.pq"
# )

# posts_file = DS.read_file('name', TableHandle, path="../paper_data/PZFlow/control/control_no_nans.pq")

# test_pdfs = testSet_posts.get_posterior(posts_file, column = 'redshift')

### Plots

In [ ]:
import h5py
import pandas as pd

postsfile = h5py.File("../paper_data/output_lsstErr_test_data_posts.hdf5")

poststable = pd.read_parquet("../paper_data/output_quantity_cut_1.pq")
poststable


In [ ]:
removeNans(poststable)

In [ ]:
poststable_inds = poststable.index.tolist()

In [ ]:
import matplotlib.pyplot as plt


for i in poststable_inds[20:40]:

    plt.plot(postsfile['meta']['xvals'][0],postsfile['data']['yvals'][i])
    plt.plot(poststable['redshift'][i] * np.ones(len(postsfile['meta']['xvals'][0])), np.linspace(0, 15, len(postsfile['meta']['xvals'][0])))
    plt.show()

In [ ]:
# import ceci 

# pr = ceci.Pipeline.read(path_lst_1[0])#parent_dir+directory+"/invz=0.33672517538070684_lsstErr_pzflow.yml")
# pr.run()

# ## 1) terminal: go to path up to invz_lsstErr_pzflow, then run these 2 lines 
# ## 2)  make list/txt file with list of paths to files made by big F

# ## do 1) 
# ## open virtual env
# ## python 
# ## import ceci 
# ## run the 2 lines of code above 


# ### at the end we can put this into a .py file that we can run at the command line 

# ## %cd ? 

# Big F's

In [ ]:
# # 'invz': invRedshift,

# deg_ls = ['BOSS', 'DEEP2', 'GAMA', 'HSC', 'VVDSf02', 'zCOSMOS', 'invz=1.0', 'invz=1.4', 'LSSTerr_only', 'LSSTerr_null', 'control']

# deg_ls_2 = ['invz=0.1', 'invz=0.6']

# deg_ls_3 = ['invz=0.3']



In [ ]:
ntrain = 1e6
seed = 17

deg_dict = {'BOSS': [specSelectBOSS(ntrain), lsstError_0(band_dict, seed)], 
            'DEEP2': [specSelectDEEP2(ntrain), lsstError_0(band_dict, seed)], 
            'GAMA': [specSelectGAMA(ntrain), lsstError_0(band_dict, seed)], 
            'HSC': [specSelectHSC(ntrain), lsstError_0(band_dict, seed)], 
            'VVDSf02': [specSelectVVDSf02(ntrain), lsstError_0(band_dict, seed)], 
            'zCOSMOS': [specSelectzCOSMOS(ntrain), lsstError_0(band_dict, seed)], 
            'invz=0.1': [invRedshift(0.1), lsstError_0(band_dict, seed)], 
            'invz=0.2': [invRedshift(0.2), lsstError_0(band_dict, seed)],
            'invz=0.3': [invRedshift(0.3), lsstError_0(band_dict, seed)], 
            'invz=0.4': [invRedshift(0.4), lsstError_0(band_dict, seed)],
            'invz=0.5': [invRedshift(0.5), lsstError_0(band_dict, seed)],
            'invz=0.6': [invRedshift(0.6), lsstError_0(band_dict, seed)],
            'invz=0.7': [invRedshift(0.7), lsstError_0(band_dict, seed)],
            'invz=0.8': [invRedshift(0.8), lsstError_0(band_dict, seed)],
            'invz=0.9': [invRedshift(0.9), lsstError_0(band_dict, seed)],
            'invz=1.0': [invRedshift(1.0), lsstError_0(band_dict, seed)], 
            'invz=1.1': [invRedshift(1.1), lsstError_0(band_dict, seed)], 
            'invz=1.2': [invRedshift(1.2), lsstError_0(band_dict, seed)], 
            'invz=1.3': [invRedshift(1.3), lsstError_0(band_dict, seed)], 
            'invz=1.4': [invRedshift(1.4), lsstError_0(band_dict, seed)], 
            'invz=1.5': [invRedshift(1.5), lsstError_0(band_dict, seed)], 
            'LSSTerr_only': [lsstError_0(band_dict, seed)], 
            'LSSTerr_null': [lsstError_null(band_dict, seed)],
            # 'control': [lsstError_control(band_dict, 39)]
            } 

# deg_dict_2 = {'invz=0.1': [invRedshift(0.1), lsstError_0(band_dict, seed)], 
#               'invz=0.6': [invRedshift(0.6), lsstError_0(band_dict, seed)]
#             }

# deg_dict_3 = {'invz=0.3': [invRedshift(0.3), lsstError_0(band_dict, seed)], 
#             }

In [ ]:
lsstError = lsstError(band_dict, 39)
lsstError.connect_input(test_data)

qcuts_2 = quantCuts_2(qcuts_dict)
qcuts_2.connect_input(lsstError)

In [ ]:
def bigF(est, deg, pathname): 
    bands = ['u','g','r','i','z','y']
    band_dict = {band: f"mag_{band}_lsst" for band in bands}
    band_dict_err = {f'mag_{band}_lsst_err':f'mag_err_{band}_lsst' for band in bands}

    inf_est_dict = {'TrainZ': [informTrainZ(deg), estimateTrainZ(deg)],
               'CMNN': [informCMNN(deg), estimateCMNN(deg)], 
               'GPz': [informGPz(deg), estimateGPz(deg)], 
               # 'PZFlow': [informPZFlow(), estimatePZFlow()], 
               'FZBoost': [informFZBoost(deg), estimateFZBoost(deg)] }
    
    if deg in ['BOSS', 'DEEP2', 'GAMA', 'HSC', 'VVDSf02', 'zCOSMOS']:
        training_data = spec_train_data
    elif 'invz' in deg:
        training_data = invz_train_data
    elif deg in ['LSSTerr_only', 'LSSTerr_null']:
        training_data = invz_train_data
    
    elif deg == 'control':
        training_data = control_train_data

    degraders = deg_dict[deg]

    informer = inf_est_dict[est][0]
    estimator = inf_est_dict[est][1]

    qcuts_1 = quantCuts_1(qcuts_dict)

    pipe = ceci.Pipeline.interactive()
    stages = [training_data, 
              test_data, 
              lsstError, 
              qcuts_1,
              qcuts_2,
              informer, 
              estimator
              ]

    if len(degraders)==2:
        selection = degraders[0]
        lsst = degraders[1]
        stages.append(selection)
        stages.append(lsst)
        selection.connect_input(training_data)
        lsst.connect_input(selection)
        qcuts_1.connect_input(lsst)

    elif len(degraders)==1:
        lsst = degraders[0]
        stages.append(lsst)
        lsst.connect_input(training_data)
        qcuts_1.connect_input(lsst)


    remapper_1 = colRemapper_1(band_dict_err)
    remapper_2 = colRemapper_2(band_dict_err)

    if est == 'CMNN' or est == 'GPz':
        stages.append(remapper_1)
        stages.append(remapper_2)

        remapper_1.connect_input(qcuts_1)
        informer.connect_input(remapper_1)

        remapper_2.connect_input(qcuts_2)
        estimator.connect_input(remapper_2, inputTag = 'input')

    else: 
        informer.connect_input(qcuts_1)
        estimator.connect_input(qcuts_2, inputTag = 'input')

    estimator.connect_input(informer, inputTag = 'model')

    for stage in stages:
        pipe.add_stage(stage)

    pipe.initialize(
    dict(model=flow_file), dict(output_dir=".", log_dir=".", resume=False), None) 

    outpath = os.path.join(pathname, deg+'_'+est+'.yml')#"% s_.yml" % deg+'_'+est)
    pipe.save(outpath)
    return outpath 
    

In [ ]:
# def bigF2(est, deg, pathname): 
#     bands = ['u','g','r','i','z','y']
#     band_dict = {band: f"mag_{band}_lsst" for band in bands}
#     band_dict_err = {f'mag_{band}_lsst_err':f'mag_err_{band}_lsst' for band in bands}

#     inf_est_dict = {'TrainZ': [informTrainZ(deg), estimateTrainZ(deg)],
#                'CMNN': [informCMNN(deg), estimateCMNN(deg)], 
#                'GPz': [informGPz(deg), estimateGPz(deg)], 
#                # 'PZFlow': [informPZFlow(), estimatePZFlow()], 
#                'FZBoost': [informFZBoost(deg), estimateFZBoost(deg)] }
    
#     if deg in ['BOSS', 'DEEP2', 'GAMA', 'HSC', 'VVDSf02', 'zCOSMOS']:
#         training_data = spec_train_data
    
#     elif deg in ['invz=1.0', 'invz=1.4','invz=0.1', 'invz=0.6', 'invz=0.3', 'LSSTerr_only']:
#         training_data = invz_train_data
    
#     elif deg == 'control':
#         training_data = control_train_data

#     degraders = deg_dict_3[deg]

#     informer = inf_est_dict[est][0]
#     estimator = inf_est_dict[est][1]

#     qcuts_1 = quantCuts_1(qcuts_dict)

#     pipe = ceci.Pipeline.interactive()
#     stages = [training_data, 
#               test_data, 
#               lsstError, 
#               qcuts_1,
#               qcuts_2,
#               informer, 
#               estimator
#               ]

#     if len(degraders)==2:
#         selection = degraders[0]
#         lsst = degraders[1]
#         stages.append(selection)
#         stages.append(lsst)
#         selection.connect_input(training_data)
#         lsst.connect_input(selection)
#         qcuts_1.connect_input(lsst)

#     elif len(degraders)==1:
#         lsst = degraders[0]
#         stages.append(lsst)
#         lsst.connect_input(training_data)
#         qcuts_1.connect_input(lsst)


#     remapper_1 = colRemapper_1(band_dict_err)
#     remapper_2 = colRemapper_2(band_dict_err)

#     if est == 'CMNN' or est == 'GPz':
#         stages.append(remapper_1)
#         stages.append(remapper_2)

#         remapper_1.connect_input(qcuts_1)
#         informer.connect_input(remapper_1)

#         remapper_2.connect_input(qcuts_2)
#         estimator.connect_input(remapper_2, inputTag = 'input')

#     else: 
#         informer.connect_input(qcuts_1)
#         estimator.connect_input(qcuts_2, inputTag = 'input')

#     estimator.connect_input(informer, inputTag = 'model')

#     for stage in stages:
#         pipe.add_stage(stage)

#     pipe.initialize(
#     dict(model=flow_file), dict(output_dir=".", log_dir=".", resume=False), None) 

#     outpath = os.path.join(pathname, deg+'_'+est+'.yml')#"% s_.yml" % deg+'_'+est)
#     pipe.save(outpath)
#     return outpath 
    

In [ ]:
xls = ['invz=0.6', 'LSSTerr_only']

In [ ]:
def makeItHelperFunc(est):
    if est != 'GPz':
        path = "../paper_data/"+est
    else: 
        path = "../paper_data/GPz_GL"
    os.chdir(path)

    path_ls = []
    for key in deg_dict: #for key in deg_dict:
        os.chdir(path)
        working_path = os.path.join(path, key)#key)
        os.makedirs(working_path, exist_ok = True)
        os.chdir(working_path)
        path_ls.append(bigF(est, key, working_path))
    
    return path_ls 

In [ ]:
# def makeItHelperFunc2(est):
#     path = "../paper_data/"+est
#     os.chdir(path)

#     path_ls = []
#     for key in deg_dict_3:
#         os.chdir(path)
#         working_path = os.path.join(path, key)
#         os.makedirs(working_path, exist_ok = True)
#         os.chdir(working_path)
#         path_ls.append(bigF2(est, key, working_path))
    
#     return path_ls 

In [ ]:
TrainZ_path_ls = makeItHelperFunc('TrainZ')

In [ ]:
CMNN_path_ls = makeItHelperFunc("CMNN")

In [ ]:
GPz_path_ls = makeItHelperFunc("GPz")

In [ ]:
FZBoost_path_ls = makeItHelperFunc("FZBoost")

In [ ]:
GPz_GL_path_ls = makeItHelperFunc('GPz')

In [ ]:
import ceci 

def runItHelperFunc(est, path_ls):
    if est != 'GPz':
        path = "../paper_data/"+est
    else: 
        path = "../paper_data/GPz_GL"
    os.chdir(path)
    ct = 0
    for key in deg_dict: 
        os.chdir(path+'/'+key)
        pr = ceci.Pipeline.read(path_ls[ct])
        pr.run()
        ct += 1

In [ ]:
# GPz GL Test 

In [ ]:
!which python



In [ ]:
from rail.core.common_params import find_rail_file

In [ ]:
!open /Users/alicec03/miniforge3/envs/rail---new/lib/python3.11/site-packages/

In [ ]:
runItHelperFunc('GPz', GPz_GL_path_ls)

In [ ]:
## TRAINZ HAS RUN! 

In [ ]:
runItHelperFunc('TrainZ', TrainZ_path_ls)

In [ ]:
runItHelperFunc('CMNN', CMNN_path_ls)

In [ ]:
runItHelperFunc('GPz', GPz_path_ls)

In [ ]:
runItHelperFunc('FZBoost', FZBoost_path_ls)

In [ ]:
lets make an error 

In [ ]:
print(np.arange(1, 11, 1))

In [ ]:
import pandas as pd

deg_file = "../paper_data/TrainZ/GAMA/output_specselection_gama.pq"
undeg_file = "../paper_data/TrainZ/GAMA/output_spec_train_set.pq"
deg = pd.read_parquet(deg_file)
orig = pd.read_parquet(undeg_file)

In [ ]:
outpath = "../paper_data/FZBoost/zCOSMOS/output_estimate_FZBoost.hdf5"

import h5py
outfile = h5py.File(outpath)

import matplotlib.pyplot as plt

x = outfile['meta']['xvals'][0]
y = outfile['data']['yvals'][0]

plt.plot(outfile['meta']['xvals'][0], outfile['data']['yvals'][3])

In [ ]:
import matplotlib.pyplot as plt


plt.scatter(orig['redshift'], orig['mag_u_lsst'])
plt.scatter(deg['redshift'], deg['mag_u_lsst'])

In [ ]:
len(deg['redshift']) 

In [ ]:
print(len(set(deg['redshift'])))

In [ ]:
cut1_path = '../paper_data/data/5e5_cut_rubin_roman_catalog.pq'
cut2_path = '../paper_data/data/1.5e6_cut_rubin_roman_catalog.pq'

cut1 = pd.read_parquet(cut1_path)
cut2 = pd.read_parquet(cut2_path)


# Visualizing Training Sets 

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

spec_set = removeNans(pd.read_parquet('../paper_data/TrainZ/BOSS/output_spec_train_set.pq') ) 
boss_deg = removeNans(pd.read_parquet('../paper_data/TrainZ/BOSS/output_quantity_cut_1.pq') ) 
deep2_deg = removeNans(pd.read_parquet('../paper_data/TrainZ/DEEP2/output_quantity_cut_1.pq') ) 
gama_deg = removeNans(pd.read_parquet('../paper_data/TrainZ/GAMA/output_quantity_cut_1.pq') ) 
hsc_deg = removeNans(pd.read_parquet('../paper_data/TrainZ/HSC/output_quantity_cut_1.pq') ) 
vvdsf02_deg = removeNans(pd.read_parquet('../paper_data/TrainZ/VVDSf02/output_quantity_cut_1.pq') ) 
zcosmos_deg = removeNans(pd.read_parquet('../paper_data/TrainZ/zCOSMOS/output_quantity_cut_1.pq') ) 

invz_set = removeNans(pd.read_parquet('../paper_data/TrainZ/invz=1.0/output_invz_train_set.pq') ) 
one_deg = removeNans(pd.read_parquet('../paper_data/TrainZ/invz=1.0/output_quantity_cut_1.pq') ) 
onefour_deg = removeNans(pd.read_parquet('../paper_data/TrainZ/invz=1.4/output_quantity_cut_1.pq') ) 
lssterr_deg = removeNans(pd.read_parquet('../paper_data/TrainZ/lsstErr_only/output_quantity_cut_1.pq'))
control_deg = removeNans(pd.read_parquet('../paper_data/TrainZ/control/output_quantity_cut_1.pq'))


In [ ]:
lssterr_deg

In [ ]:
import matplotlib
matplotlib.rcParams.update({'font.size':22})

import matplotlib.pyplot as plt

fig, axes = plt.subplots(nrows = 6, ncols = 10, figsize = (120, 40))
bands = ['u', 'g', 'r', 'i', 'z', 'y']
ct = 0
for band in bands: 
    axes[ct][0].scatter(spec_set['redshift'], spec_set['mag_'+band+'_lsst'], s=0.5)
    axes[ct][0].scatter(boss_deg['redshift'], boss_deg['mag_'+band+'_lsst'], s=0.5)
    # axes[ct][0].hist2d(boss_deg['redshift'], boss_deg['mag_'+band+'_lsst'], bins = (np.linspace(0, 3, 100), np.linspace(15, 28, 100)), density = True)

    axes[ct][1].scatter(spec_set['redshift'], spec_set['mag_'+band+'_lsst'], s=0.5)
    axes[ct][1].scatter(deep2_deg['redshift'], deep2_deg['mag_'+band+'_lsst'], s=0.5)
    # axes[ct][1].hist2d(deep2_deg['redshift'], deep2_deg['mag_'+band+'_lsst'], bins = (np.linspace(0, 3, 100), np.linspace(15, 28, 100)), density = True)

    axes[ct][2].scatter(spec_set['redshift'], spec_set['mag_'+band+'_lsst'], s=0.5)
    axes[ct][2].scatter(gama_deg['redshift'], gama_deg['mag_'+band+'_lsst'], s=0.5)
    # axes[ct][2].hist2d(gama_deg['redshift'], gama_deg['mag_'+band+'_lsst'], bins = (np.linspace(0, 3, 100), np.linspace(15, 28, 100)), density = True)

    axes[ct][3].scatter(spec_set['redshift'], spec_set['mag_'+band+'_lsst'], s=0.5)
    axes[ct][3].scatter(hsc_deg['redshift'], hsc_deg['mag_'+band+'_lsst'], s=0.5)
    # axes[ct][3].hist2d(hsc_deg['redshift'], hsc_deg['mag_'+band+'_lsst'], bins = (np.linspace(0, 3, 100), np.linspace(15, 28, 100)), density = True)

    axes[ct][4].scatter(spec_set['redshift'], spec_set['mag_'+band+'_lsst'], s=0.5)
    axes[ct][4].scatter(vvdsf02_deg['redshift'], vvdsf02_deg['mag_'+band+'_lsst'], s=0.5)
    # axes[ct][4].hist2d(vvdsf02_deg['redshift'], vvdsf02_deg['mag_'+band+'_lsst'], bins = (np.linspace(0, 3, 100), np.linspace(15, 28, 100)), density = True)

    axes[ct][5].scatter(spec_set['redshift'], spec_set['mag_'+band+'_lsst'], s=0.5)
    axes[ct][5].scatter(zcosmos_deg['redshift'], zcosmos_deg['mag_'+band+'_lsst'], s=0.5)
    # axes[ct][5].hist2d(zcosmos_deg['redshift'], zcosmos_deg['mag_'+band+'_lsst'], bins = (np.linspace(0, 3, 100), np.linspace(15, 28, 100)), density = True)


    axes[ct][6].scatter(invz_set['redshift'], invz_set['mag_'+band+'_lsst'], s=0.5)
    axes[ct][6].scatter(one_deg['redshift'], one_deg['mag_'+band+'_lsst'], s=0.5)
    # axes[ct][6].hist2d(one_deg['redshift'], one_deg['mag_'+band+'_lsst'], bins = (np.linspace(0, 3, 100), np.linspace(15, 28, 100)), density = True)

    axes[ct][7].scatter(invz_set['redshift'], invz_set['mag_'+band+'_lsst'], s=0.5)
    axes[ct][7].scatter(onefour_deg['redshift'], onefour_deg['mag_'+band+'_lsst'], s=0.5)
    # axes[ct][7].hist2d(onefour_deg['redshift'], onefour_deg['mag_'+band+'_lsst'], bins = (np.linspace(0, 3, 100), np.linspace(15, 28, 100)), density = True)

    axes[ct][8].scatter(invz_set['redshift'], invz_set['mag_'+band+'_lsst'], s=0.5)
    axes[ct][8].scatter(lssterr_deg['redshift'], lssterr_deg['mag_'+band+'_lsst'], s=0.5)
    # axes[ct][8].hist2d(lssterr_deg['redshift'], lssterr_deg['mag_'+band+'_lsst'], bins = (np.linspace(0, 3, 100), np.linspace(15, 28, 100)), density = True)

    axes[ct][9].scatter(invz_set['redshift'], invz_set['mag_'+band+'_lsst'], s=0.5)
    axes[ct][9].scatter(control_deg['redshift'], control_deg['mag_'+band+'_lsst'], s=0.5)
    # axes[ct][9].hist2d(control_deg['redshift'], control_deg['mag_'+band+'_lsst'], bins = (np.linspace(0, 3, 100), np.linspace(15, 28, 100)), density = True)


    ct += 1

axes[0][0].set_title("BOSS", size = 'large')
axes[0][1].set_title("DEEP2", size = 'large')
axes[0][2].set_title("GAMA", size = 'large')
axes[0][3].set_title("HSC", size = 'large')
axes[0][4].set_title("VVDSf02", size = 'large')
axes[0][5].set_title("zCOSMOS", size = 'large')
axes[0][6].set_title("Inverse Redshift Incompleteness, $z_0 = 1.0$", size = 'large')
axes[0][7].set_title("Inverse Redshift Incompleteness, $z_0 = 1.4$", size = 'large')
axes[0][8].set_title("LSST Error Model Only", size = 'large')
axes[0][9].set_title("Control", size = 'large')

matplotlib.rcParams.update({'font.size':30})

axes[0][0].set_ylabel('u band', size = 'large')
axes[1][0].set_ylabel('g band', size = 'large')
axes[2][0].set_ylabel('r band', size = 'large')
axes[3][0].set_ylabel('i band', size = 'large')
axes[4][0].set_ylabel('z band', size = 'large')
axes[5][0].set_ylabel('y band', size = 'large')

fig.suptitle("Training Sets", size = "xx-large")
fig.supxlabel("Redshift", size = 'x-large')
fig.supylabel("Magnitude", size = 'x-large')

# Visualizing Estimator Outputs

In [ ]:
def gaussian(file, grid, ct):
    return (1. /(file['data']['scale'][ct] * np.sqrt(2*np.pi))) * np.exp((-1/2) * ((grid - file['data']['loc'][ct])/ file['data']['scale'][ct] )**2)

In [ ]:
import h5py
import pandas as pd
import matplotlib.pyplot as plt

def plotByDeg(ax, deg, galaxy):

    import h5py
    import pandas as pd
    import matplotlib.pyplot as plt
    true_point_file = pd.read_parquet("../paper_data/output_quantity_cut_1.pq")['redshift']

    gals = true_point_file.index.tolist()
    gal = gals[galaxy]

    true_point_grid = np.arange(0, 31, 1)
    true_point_data = true_point_file[gal] * np.ones(31)

    true_posts_file = h5py.File("../paper_data/output_lsstErr_test_data_posts.hdf5")
    true_posts_grid = true_posts_file['meta']['xvals'][0]
    true_posts_data = true_posts_file['data']['yvals'][gal]

    trainz_file  = h5py.File("../paper_data/TrainZ/"+deg+"/output_estimate_TrainZ.hdf5")
    trainz_grid = trainz_file['meta']['xvals'][0]
    trainz_data = trainz_file['data']['yvals'][gal]
    cmnn_file = h5py.File("../paper_data/CMNN/"+deg+"/output_estimate_CMNN.hdf5")
    cmnn_grid = np.arange(0.0, 3.01, 0.01)
    cmnn_data = gaussian(cmnn_file, cmnn_grid, gal)
    gpz_file = h5py.File("../paper_data/GPz/"+deg+"/output_estimate_GPz.hdf5")
    gpz_grid = cmnn_grid
    gpz_data = gaussian(gpz_file, gpz_grid, gal)
    fzboost_file = h5py.File("../paper_data/FZBoost/"+deg+"/output_estimate_FZBoost.hdf5")
    fzboost_grid = fzboost_file['meta']['xvals'][0]
    fzboost_data = fzboost_file['data']['yvals'][gal]
    pzflow_file = h5py.File("../paper_data/PZFlow/"+deg+"/output_estimate_pzflow.hdf5")
    pzflow_grid = pzflow_file['meta']['xvals'][0]
    pzflow_data = pzflow_file['data']['yvals'][gal]

    ax.plot(true_posts_grid, true_posts_data, color = 'k', label = 'true posterior')
    ax.plot(true_point_data, true_point_grid, color = 'k', linestyle = 'dashed', label = 'true redshift')
    ax.plot(trainz_grid, trainz_data, label = 'TrainZ')
    ax.plot(cmnn_grid, cmnn_data, label = 'CMNN')
    ax.plot(gpz_grid, gpz_data, label = 'GPz')
    ax.plot(fzboost_grid, fzboost_data, label = 'FZBoost')
    ax.plot(pzflow_grid, pzflow_data, label = 'PZFlow')
    # ax.plot([], [], label = 'PZFlow')

    ax.set_title("Estimator outputs for galaxy #"+str(gal))
    ax.set_xlabel('Redshift')
    ax.set_ylabel("Output PDF")
    ax.legend()

In [ ]:
# plotByDeg('invz=1.0', 9)

In [ ]:
def makeDegCornerPlot(dim, deg):
    fig, axes = plt.subplots(nrows = dim, ncols = dim, figsize = (25, 25))
    ct = 0

    for i in range(0, dim):
        for j in range(0, dim):
            plotByDeg(axes[i][j], deg, ct)
            ct += 1
    
    if deg == 'invz=1.0':
        deg_title = "Inverse Redshift Incompleteness with $z_0 = 1.0$ + Default LSST Error Model"
    elif deg == 'invz=1.4':
        deg_title = "Inverse Redshift Incompleteness with $z_0 = 1.4$ + Default LSST Error Model"
    elif deg == 'LSSTerr_only': 
        deg_title = "Default LSST Error Model" 
    elif deg == 'LSSTerr_null':
        deg_title = 'No Degradation'
    elif deg == 'control':
        deg_title = 'Trained on Test Set'
    else: 
        deg_title = deg+" Spectrascopic Survey Degradation + Default LSST Error Model"
    fig.suptitle("Training Set Degradation: "+deg_title, size = 'x-large' )

In [ ]:
makeDegCornerPlot(5, 'control')

In [ ]:
import h5py
import matplotlib.pyplot as plt
matplotlib.rcParams.update({'font.size':10})

TrainZpath = "../paper_data/TrainZ/DEEP2/output_estimate_TrainZ.hdf5"
TrainZfile = h5py.File(TrainZpath)
CMNNpath = "../paper_data/CMNN/DEEP2/output_estimate_CMNN.hdf5"
CMNNfile = h5py.File(CMNNpath)
GPzpath = "../paper_data/GPz/DEEP2/output_estimate_GPz.hdf5"
GPzfile = h5py.File(GPzpath)
FZBoostpath = "../paper_data/FZBoost/DEEP2/output_estimate_FZBoost.hdf5"
FZBoostfile = h5py.File(FZBoostpath)
truth_pdfs = h5py.File("../paper_data/output_lsstErr_test_data_posts.hdf5")
point_zs = pd.read_parquet("../paper_data/output_lsst_error.pq")

grid = np.linspace(0, 3, 101)


for gal in range(0, 20):
    plt.plot(TrainZfile['meta']['xvals'][0], TrainZfile['data']['yvals'][gal], label = "TrainZ")
    plt.plot(grid, gaussian(CMNNfile, grid, gal), label = "CMNN")


    #plt.plot(grid, gaussian(GPzfile, grid, gal))
    #plt.plot(FZBoostfile['meta']['xvals'][0], FZBoostfile['data']['yvals'][gal])

    plt.plot(truth_pdfs['meta']['xvals'][0], truth_pdfs['data']['yvals'][gal], label = "true pdf")
    plt.plot(point_zs['redshift'][gal] * np.ones(len(truth_pdfs['meta']['xvals'][0])), np.linspace(0, 15, len(truth_pdfs['meta']['xvals'][0])), label = "true point z")

    plt.legend()
    plt.show()

In [ ]:
## we'll want to compare PITs of posteriors with PITs of estimates 